In [56]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import VotingClassifier

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("regression_dataset.csv")

df.head()
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,50276.0,27.551416,5.964977,20.00,23.00,26.00,30.00,84.00
income,50276.0,66677.871151,39316.425504,4200.00,45000.00,60000.00,80000.00,1900000.00
emplyment_length,50276.0,4.866159,4.001135,0.00,2.00,4.00,7.00,150.00
loan_amount,50276.0,8887.882528,5327.333853,500.00,5000.00,8000.00,12000.00,35000.00
loan_interest_rate,50276.0,10.258792,2.827098,5.42,7.51,10.39,12.42,22.11
loan_income_ratio,50276.0,0.145096,0.077869,0.00,0.08,0.13,0.20,0.63
credit_history_length,50276.0,5.818482,4.002464,2.00,3.00,4.00,8.00,30.00
max_allowed_loan,50276.0,81390.696913,58124.883347,232.00,49105.50,69418.50,98987.00,2638778.00


In [11]:
x = df.drop(columns=['max_allowed_loan'])
y = df['max_allowed_loan']

In [12]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)

In [16]:
cat_cols = x_train.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

In [17]:
x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)

In [20]:
tree_full = DecisionTreeRegressor(random_state=42)
tree_full.fit(x_train, y_train)

y_pred_full = tree_full.predict(x_test)

In [43]:
print("Full Tree")
print("MAE:", mean_absolute_error(y_test, y_pred_full))
print("RMSE:", root_mean_squared_error(y_test, y_pred_full))
print("R2:", r2_score(y_test, y_pred_full))

Full Tree
MAE: 1266.8939936356405
RMSE: 8203.729478560259
R2: 0.9751560781312351


In [38]:
tree_pruned = DecisionTreeRegressor(max_depth=8, random_state=42)
tree_pruned.fit(x_train, y_train)

y_pred_pruned = tree_pruned.predict(x_test)

In [44]:
print("Pruned Tree")
print("MAE:", mean_absolute_error(y_test, y_pred_pruned))
print("RMSE:", root_mean_squared_error(y_test, y_pred_pruned))
print("R2:", r2_score(y_test, y_pred_pruned))

Pruned Tree
MAE: 4861.8284994411315
RMSE: 11448.992519061863
R2: 0.9516126279215947


In [53]:
df_class = pd.read_csv("classification_dataset.csv")

x_new = df_class.drop(columns=['loan_approval_status'])
y_new = df_class['loan_approval_status']

In [59]:
x_train_c, x_test_c, y_train_c, y_test_c = train_test_split(
    x_new, y_new,
    test_size=0.2,
    stratify=y_new,
    random_state=42
)

In [60]:
#Seperating the columns as numerical and categorical for scalng and encoding

num_cols_new = x_train_c.select_dtypes(include=['int64', 'float64']).columns
cat_cols_new = x_train_c.select_dtypes(include=['object']).columns

In [61]:
#Making a pipeline with both scaler and encoder and saving them as transformers
preprocessor_new = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_new),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols_new)
    ]
)

In [65]:
# reuse ColumnTransformer
x_train_c = preprocessor_new.fit_transform(x_train_c)
x_test_c = preprocessor_new.transform(x_test_c)

In [66]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

lr = LogisticRegression(max_iter=1000)
nb = GaussianNB()
knn = KNeighborsClassifier(n_neighbors=9)

In [67]:
voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr),
        ('nb', nb),
        ('knn', knn)
    ],
    voting='hard'
)

In [68]:
voting_clf.fit(x_train_c, y_train_c)

,estimators,"[('lr', ...), ('nb', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [69]:
y_pred_vote = voting_clf.predict(x_test_c)

In [70]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test_c, y_pred_vote))
print(classification_report(y_test_c, y_pred_vote))

[[9732  324]
 [ 698  971]]
              precision    recall  f1-score   support

           0       0.93      0.97      0.95     10056
           1       0.75      0.58      0.66      1669

    accuracy                           0.91     11725
   macro avg       0.84      0.77      0.80     11725
weighted avg       0.91      0.91      0.91     11725

